# Step 2 -- Propagation

In this notebook, we will propagate the annotations that we formatted in the previous notebook. To do this, we will use the HOGProp tool.

In [1]:
mini_dataset_path = '/work/FAC/FBM/DBC/cdessim2/default/summer_school2026/mini_dataset'

## Running HOGPROP

HOGPROP requires three inputs:

- A GO ontology file (`go-basic.obo`)
- A GAF file containing extant annotations
- An OrthoXML file containing HOGs generated by FastOMA

The command below propagates GO annotations through the HOG hierarchy and stores the results in an HDF5 file.

We can run the hogprop tool with this command, however even with this small dataset this is quite slow.

In [2]:
import subprocess

cmd = [
    "hogprop",
    "--obo", "../step1_annotation/go-basic.obo",
    "--gaf", "../step1_annotation/extant_predictions.gaf.gz",
    "--go_filter", "all",
    "--combination_func", "max",
    "--oxml", f"{mini_dataset_path}/fastoma_output/FastOMA_HOGs.orthoxml",
    "--results", "hogprop_results.h5"
]

print(" ".join(cmd))
subprocess.run(cmd)

hogprop --obo ../step1_annotation/go-basic.obo --gaf ../step1_annotation/extant_predictions.gaf.gz --go_filter all --combination_func max --oxml /work/FAC/FBM/DBC/cdessim2/default/summer_school2026/mini_dataset/fastoma_output/FastOMA_HOGs.orthoxml --results hogprop_results.h5


Loading species: 10 species [00:07,  1.40 species/s]
Loading Annotations: 17676829it [00:51, 342356.02it/s]
Propagating through HOGs: 221it [04:04,  1.03s/it]

KeyboardInterrupt: 

---

The example above is suitable for small datasets and for understanding how HOGPROP works. For larger datasets, HogProp is typically run on a compute cluster using a job scheduler such as SLURM.

The script to run HogProp on all the proteomes in parallel on an HPC is `sbatch hogprop.sh` in this directory.

Then, once all jobs are complete we can run `source hogprop-convert.sh` in order to convert to a single output database. This can then be used in the next step.

## Questions

For larger datasets, HOGPROP is usually submitted to a cluster.

Open `hogprop.sh` and `hogprop-convert.sh` and identify:

1. Which file contains the GO ontology?
2. Which file contains the GAF annotations?
3. Which file contains the HOG structure?
4. Where are the results written?
5. How many HOGs have annotations?

1. Which file contains the GO ontology?

`go-basic.obo`

2. Which file contains the GAF annotations?

`extant_predictions.gaf.gz`

3. Which file contains the HOG structure?

`FastOMA_HOGs.orthoxml`

4. Where are the *final* results written?

`hogprop_output.h5`

5. How many HOGs have annotations?

In [12]:
#How many HOG annotation records were generated?

import tables

h5 = tables.open_file("hogprop_output.h5")

print(
    h5.root.HOGAnnotations.GeneOntology.nrows,
    "HOG annotation records"
)

h5.close()

11799967 HOG annotation records


In [14]:
#How many unique HOGs received at least one GO annotation?

import numpy as np

h5 = tables.open_file("hogprop_output.h5")

table = h5.root.HOGAnnotations.GeneOntology

hog_ids = table.col("HogNr")

print("Unique HOGs:", len(np.unique(hog_ids)))

h5.close()

Unique HOGs: 100872
